<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# A learned ranker

**[A learned ranker](https://learning.nextia-ai.com/courses/recommendations/m07/a-learned-ranker/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 7, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** describe candidates with features, train a LightGBM LambdaMART ranker on the two weeks before the test period, and compare it with trending.

| | |
|---|---|
| **Time** | About 25 minutes |
| **Needs** | An internet connection for the install and setup cells (LightGBM, and about 7 MB of data). No account and no key. |
| **You should know** | Candidates, from [Why two stages](https://learning.nextia-ai.com/courses/recommendations/m07/why-two-stages/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1, matplotlib 3.11.2 and LightGBM 4.7.0, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m07/a-learned-ranker/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M07-L02-a-learned-ranker/a-learned-ranker-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, scipy 1.18.1, lightgbm 4.7.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2 lightgbm==4.7.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy', 'lightgbm']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 7 MB) into a folder `larkfield-recs` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-recs` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop, and its shop log is **synthetic**: our code made the products, the customers and every view, basket and order, with a model of customer behaviour. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/data/dataset-card.md) explains how.

In [ ]:
# Setup: download the files into larkfield-recs/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C37/"
FILES = {
    "data/products.csv": "3da37a3809f6067da12eb5778b8abcf809545599db6959208c72a790861407ff",
    "data/events.csv.gz": "3f46ddfff683586ab1548bc0e6756d85a439a1de941e6de3030edc383d06d1b3",
    "data/splits.json": "c58597803ced6fa787ff23c86d8ee6e9f0c1d7024c0ec109d2cca25caf6213e6",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-recs").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-recs and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-recs.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, SciPy's sparse matrices as `sp` and matplotlib. It defines the course's small helpers, the same ideas as the course project:

- `products()`, `customers()` and `events()` read the data. `events()` has one row per view, basket add or order, with the column `day` (days since 1 February 2025).
- `split(period)` gives the events before the period (`train`, to learn from) and, for each customer who ordered in the period, the products they ordered (`truth`). The periods are `"validation"`, `"test"` and `"launch"`.
- `recall_at_k(list, ordered, k)`, `ndcg_at_k(list, ordered, k)` and `coverage(lists, n_products)` measure suggestion lists.
- `show(ids)` prints products with their title and subcategory.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the time split, the measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
START = pd.Timestamp("2025-02-01")

def products():
    """One row per product (2,994)."""
    df = pd.read_csv("data/products.csv", keep_default_na=False)
    for col in ("age_restricted", "sensitive", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def customers():
    """One row per customer (40,000)."""
    return pd.read_csv("data/customers.csv")

def events():
    """One row per event: view, basket or order. day = days since 1 February 2025."""
    ev = pd.read_csv("data/events.csv.gz", parse_dates=["timestamp"])
    ev["day"] = (ev["timestamp"] - START).dt.days
    return ev

def split(period, ev=None):
    """train = the events before the period; truth = {customer: set of products ordered in the period}."""
    ev = events() if ev is None else ev
    first, last = json.load(open("data/splits.json"))[period]
    a, b = (pd.Timestamp(first) - START).days, (pd.Timestamp(last) - START).days
    train = ev[ev["day"] < a]
    target = ev[(ev["day"] >= a) & (ev["day"] <= b) & (ev["event"] == "order")]
    truth = target.groupby("customer_id")["product_id"].agg(set).to_dict()
    return train, truth

def recall_at_k(rec, ordered, k=10):
    """Share of the ordered products that are in the first k suggestions."""
    return len(set(list(rec)[:k]) & set(ordered)) / len(ordered)

def ndcg_at_k(rec, ordered, k=10):
    """DCG of the list divided by the best possible DCG (discount log2(place + 1))."""
    gains = [1.0 if p in ordered else 0.0 for p in list(rec)[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(ordered), k)))
    return dcg / ideal

def coverage(lists, n_products, k=10):
    """Share of the catalogue that appears in at least one list."""
    return len({p for rec in lists for p in list(rec)[:k]}) / n_products

def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def show(ids, n=10):
    """Print products: place, ID, title, subcategory."""
    global _PR
    if "_PR" not in globals():
        _PR = products().set_index("product_id")
    for i, p in enumerate(list(ids)[:n], 1):
        print(f"{i:2}. {p}  {_PR.loc[p, 'title'][:52]:<52}  {_PR.loc[p, 'subcategory']}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Which mistake costs more?

A ranker that learns with `lambdarank` pushes apart each pair where an unordered product is above an ordered one, harder when the swap changes NDCG more. Here is one customer with one ordered product.

> **Predict.** Is it worth more to move the ordered product from place 2 to place 1, or from place 4 to place 3? How many times more?

In [ ]:
# A tiny example we made: one customer, one ordered product.
# NDCG gives a found product at place k the value 1 / log2(k + 1).
def value(place):
    return 1 / np.log2(place + 1)
gain_top = value(1) - value(2)     # the ordered product moves from place 2 to place 1
gain_low = value(3) - value(4)     # the ordered product moves from place 4 to place 3
print(f"place 2 -> 1: +{gain_top:.3f}   place 4 -> 3: +{gain_low:.3f}   ratio {gain_top / gain_low:.1f}")

## 2. Candidates, features and labels

The project's ranker has five sources and 26 features. This notebook builds a small version: two sources (trending and the customer's recent products) and six features. Every row is one candidate of one customer. The label is 1 if the customer ordered it in the 14 days after the cut-off.

For training, the cut-off is 14 days before the test period, so the labels are the orders of the two weeks before it. The test period is never used for training. It takes about 10 seconds.

If `import lightgbm` fails on a Mac with a message about `libomp`, run `brew install libomp` in a terminal, restart the kernel and run the notebook again.

In [ ]:
import lightgbm as lgb
ev = events()
pr = products()
category = pr.set_index("product_id")["category"]
TEST_DAY = (pd.Timestamp(json.load(open("data/splits.json"))["test"][0]) - START).days
FEATURES = ["trending_rank", "recent_rank", "buyers_7", "days_since_touch", "cat_share", "bought_before"]

def rows(cutoff):
    """Candidates and six features from the events before `cutoff` (a day number);
    labels = the orders of the 14 days from `cutoff`, for the customers who ordered then."""
    before = ev[ev["day"] < cutoff]
    o = before[before["event"] == "order"]
    window = ev[(ev["day"] >= cutoff) & (ev["day"] < cutoff + 14) & (ev["event"] == "order")]
    custs = np.sort(window["customer_id"].unique())
    # source 1: trending = the 100 products with the most buyers in the last 28 days
    b28 = o[o["day"] >= cutoff - 28].groupby("product_id")["customer_id"].nunique()
    trend = b28.sort_values(ascending=False, kind="stable").index[:100]
    cand = pd.DataFrame({"customer_id": np.repeat(custs, len(trend)), "product_id": np.tile(trend, len(custs)),
                         "trending_rank": np.tile(np.arange(1, len(trend) + 1), len(custs))})
    # source 2: recent = the customer's own products of the last 30 days, newest first (at most 30)
    mine = before[before["customer_id"].isin(custs)]
    last = mine.groupby(["customer_id", "product_id"])["day"].max().rename("last_day").reset_index()
    rec = last[last["last_day"] >= cutoff - 30].sort_values(["customer_id", "last_day"], ascending=[True, False])
    rec["recent_rank"] = rec.groupby("customer_id").cumcount() + 1
    rec = rec[rec["recent_rank"] <= 30][["customer_id", "product_id", "recent_rank"]]
    cand = cand.merge(rec, on=["customer_id", "product_id"], how="outer")
    cand[["trending_rank", "recent_rank"]] = cand[["trending_rank", "recent_rank"]].fillna(999)
    # the product: buyers in the last 7 days
    b7 = o[o["day"] >= cutoff - 7].groupby("product_id")["customer_id"].nunique()
    cand["buyers_7"] = np.log1p(cand["product_id"].map(b7).fillna(0))
    # the pair: days since the customer touched it, their share of events in its category, bought before
    cand = cand.merge(last, on=["customer_id", "product_id"], how="left")
    cand["days_since_touch"] = (cutoff - cand["last_day"]).fillna(999).clip(upper=999)
    per_cat = mine.assign(cat=mine["product_id"].map(category)).groupby(["customer_id", "cat"]).size()
    total = mine.groupby("customer_id").size()
    key = pd.MultiIndex.from_arrays([cand["customer_id"], cand["product_id"].map(category)])
    cand["cat_share"] = (per_cat.reindex(key).fillna(0).to_numpy()
                         / cand["customer_id"].map(total).fillna(1).clip(lower=1).to_numpy())
    bought = o[["customer_id", "product_id"]].drop_duplicates().assign(bought_before=1)
    cand = cand.merge(bought, on=["customer_id", "product_id"], how="left").fillna({"bought_before": 0})
    # the label: 1 if ordered in the 14 days from the cutoff
    lab = window[["customer_id", "product_id"]].drop_duplicates().assign(label=1)
    cand = cand.merge(lab, on=["customer_id", "product_id"], how="left").fillna({"label": 0})
    truth = window.groupby("customer_id")["product_id"].agg(set).to_dict()
    return cand.sort_values(["customer_id", "product_id"]).reset_index(drop=True), truth

train_rows, _ = rows(TEST_DAY - 14)          # the two weeks before the test period
print(f"{train_rows['customer_id'].nunique():,} customers, {len(train_rows):,} candidate rows, "
      f"{int(train_rows['label'].sum()):,} ordered (label 1)")
print(train_rows[train_rows["customer_id"] == train_rows["customer_id"].iloc[0]].head(5)[FEATURES + ["label"]])

## 3. Train the ranker

Each customer's candidates form one **group**: the ranker only compares candidates of the same customer. We keep every ordered row and 50 random unordered rows per customer (negative sampling), then train 100 trees on one thread.

In [ ]:
# Keep the customers with at least one ordered candidate; keep every label-1 row and 50 random label-0 rows each.
rng = np.random.default_rng(37)
t = train_rows[train_rows.groupby("customer_id")["label"].transform("max") == 1].copy()
t["r"] = rng.random(len(t))
t["neg_rank"] = t[t["label"] == 0].groupby("customer_id")["r"].rank(method="first")
t = t[(t["label"] == 1) | (t["neg_rank"] <= 50)].sort_values("customer_id")
groups = t.groupby("customer_id", sort=True).size().to_numpy()   # one group per customer, in the same order
params = {"objective": "lambdarank", "learning_rate": 0.05, "num_leaves": 15, "min_data_in_leaf": 20,
          "seed": 37, "deterministic": True, "num_threads": 1, "verbose": -1}
ranker = lgb.train(params, lgb.Dataset(t[FEATURES], label=t["label"], group=groups), num_boost_round=100)
gain = ranker.feature_importance("gain")
print(f"trained on {len(groups):,} customers, {len(t):,} rows")
for name, g in sorted(zip(FEATURES, gain / gain.sum()), key=lambda x: -x[1]):
    print(f"  {name:<18}{g:6.1%}")

## 4. Your turn: rank the test period

The next cell builds the test customers' candidates from the events before the test period, and scores trending alone. Then it is your turn: score every test candidate with the ranker, sort each customer's candidates by score, and compute recall@10.

In [ ]:
test_rows, truth = rows(TEST_DAY)            # candidates and features from the events before the test period
print(f"{len(truth):,} test customers, {len(test_rows):,} candidate rows")

def recall_of(df, column, ascending):
    """Mean recall@10 when each customer's candidates are sorted by `column`."""
    top = df.sort_values(["customer_id", column], ascending=[True, ascending]).groupby("customer_id").head(10)
    lists = top.groupby("customer_id")["product_id"].agg(list)
    return np.mean([recall_at_k(lists.get(c, []), truth[c]) for c in truth])

recall_trending = recall_of(test_rows, "trending_rank", True)
print(f"trending alone: recall@10 {recall_trending:.3f}")

In [ ]:
test_rows["score"] = ...   # the ranker's score for every test row (ranker.predict on the six features)
recall_ranker = ...        # recall@10 when each customer's candidates are sorted by score, highest first (use recall_of)
print(recall_ranker)

In [ ]:
expect('recall_ranker', rounded(recall_ranker, 3), 0.227, tolerance=0.005)

> **Check.** On the same candidates, trending alone gets 0.156 and the small ranker 0.227. Most of the gain comes from mixing two clues: what sells this week, and what the customer touched recently. The project's ranker, with five sources and 26 features, gets 0.234 (`recs evaluate two-stage`): the other sources and features add a little more.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Business rules and re-ranking](https://learning.nextia-ai.com/courses/recommendations/m07/business-rules-and-re-ranking/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m07/a-learned-ranker/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield shop log is synthetic: our code made the products, the customers and everything they did, with a model of customer behaviour. No real customer is in this data.